In [1]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
print(sys.executable)

e:\testing\agentenv\Scripts\python.exe


In [15]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain import hub
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import requests

In [16]:
import langchain
print(langchain.__version__)

0.1.16


In [17]:
from langchain.agents import create_react_agent, AgentExecutor

In [18]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
load_dotenv(dotenv_path='./.env')

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]
WEATHERSTACK_API_KEY = os.environ["WEATHERSTACK_API_KEY"]

In [19]:
search_tool = TavilySearchResults(max_results=2)

In [20]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [21]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-april-2026',
  'content': '## Bullet points\n\n Check out "The latest AI news we announced in April" for Google\'s newest tech updates.\n Google Cloud introduced powerful new tools and chips to help businesses build AI agents.\n You can now create professional videos for free using the new Google Vids suite.\n New coding tools like Learn Mode in Colab act as your personal programming tutor.\n Google is using AI to improve healthcare access and help students with test prep.\n\n## Basic explainer [...] Learn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\n# The latest AI news we announced in April 2026\n\nMay 04, 2026\n\nHere’s a recap of our biggest AI updates from April, including Gemma 4, Deep Research Max and all the big announcements from Cloud Next ‘26.\n\n## General summary\n\nGoogle’s April updates focused on the "agentic era," introducing the Gemini Enterprise Agen

In [22]:
import os

from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=GROQ_API_KEY,
        
    model="llama-3.3-70b-versatile",
    temperature=0
)


In [23]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI program go on a diet?\n\nBecause it wanted to lose some bytes.', response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 41, 'total_tokens': 60, 'completion_time': 0.035063816, 'completion_tokens_details': None, 'prompt_time': 0.002220503, 'prompt_tokens_details': None, 'queue_time': 0.049119797, 'total_time': 0.037284319}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'finish_reason': 'stop', 'logprobs': None}, id='run-3f5131ef-4d6e-4b05-b049-c42b74897d97-0')

In [24]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [25]:
prompt = hub.pull("hwchase17/react")

In [26]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
    
)

In [28]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [30]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of tamil nadu"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
Thought: To find the capital of Tamil Nadu, I can use my general knowledge or search for it. Since I'm not sure if my general knowledge is up-to-date, I'll use the tavily_search_results_json to find the capital of Tamil Nadu.

Action: tavily_search_results_json
Action Input: capital of Tamil Nadu[{'url': 'https://en.wikipedia.org/wiki/Tamil_Nadu', 'content': "Chennai is the capital of the state and houses the state executive, legislative and head of judiciary. The administration of the state government functions through various secretariat departments. There are 43 departments of the state and the departments have further sub-divisions which may govern various undertakings and boards. The state is divided into 38 districts, each of which is administered by a District Collector, who is an officer of the Indian Administrative Service (IAS) appointed to the district by the Government of Tamil Nadu. For land revenue administration, the districts are 